# Toy Human Microbiome Project analysis
This notebook contains the analysis of the 49 samples from the CAMI 2 THMP dataset. All the samples were subject to the entire MOSHPIT MAG reconstruction pipeline available in QIIME 2. Here, we focus on downstream analysis of the resulting feature tables.

In [ ]:
# silence pandas' warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import qiime2 as q2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import glob
import os
import re
import shutil
import time

from __future__ import annotations
from matplotlib.colors import LinearSegmentedColormap
from qiime2.plugins import feature_table, quality_control, taxa as q2_taxa
from typing import List, Dict, Iterable, Optional
from Bio import Entrez

%matplotlib inline

## Gold Standard Assembly data preparation
We need to prepare a set of QIIME2-compatible feature tables which we will use later for abundance comparisons.

In [ ]:
gsa_data_dir = ""
email = ""

In [ ]:
directories = {
    "airways": {"short": "air", "group": "asug"},
    "gastrointestinal": {"short": "gastro", "group": "go"},
    "oral": {"short": "oral", "group": "go"},
    "skin": {"short": "skin", "group": "asug"},
    "urogenital": {"short": "uro", "group": "asug"}
}

RANK_PREFIX = {
    "domain": "d__",
    "kingdom": "k__",
    "phylum": "p__",
    "class": "c__",
    "order": "o__",
    "family": "f__",
    "genus": "g__",
    "species": "s__",
    "subspecies": "ssp__"
}

In [ ]:
def read_metadata(path: str):
    df = pd.read_csv(path, sep="\t", index_col=0)
    return df
    
def read_abundance(path: str, sample: str):
    df = pd.read_csv(path, sep="\t", index_col=0, header=None)
    df.columns = [sample]
    return df.transpose()

def _chunked(xs: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(xs), n):
        yield xs[i : i + n]


def fetch_taxonomy_series(
    taxids: Iterable[int | str],
    *,
    email: str,
    api_key: Optional[str] = None,
    batch_size: int = 200,
    sleep_s: float = 0.34,  # ~3 req/s without API key
    sep: str = "; ",
    include_rank: bool = False,
    include_self: bool = True,
) -> pd.Series:
    """
    Fetch taxonomy lineage strings from NCBI Taxonomy for TaxIDs.

    Parameters
    ----------
    taxids : iterable of int|str
        NCBI Taxonomy IDs.
    email : str
        Required by NCBI Entrez.
    api_key : str, optional
        NCBI API key (higher rate limits).
    batch_size : int
        Number of TaxIDs per efetch request.
    sleep_s : float
        Delay between requests to be polite / avoid throttling.
    sep : str
        Separator used in taxonomy strings.
    include_rank : bool
        If True, format each node like "rank:name" (e.g. "phylum:Firmicutes").
    include_self : bool
        If True, include the queried taxon at the end of the lineage.

    Returns
    -------
    pandas.Series
        Index: TaxID (int), Values: taxonomy lineage strings (str) or NA if missing.
    """
    Entrez.email = email
    if api_key:
        Entrez.api_key = api_key

    # Normalise & deduplicate while preserving order
    seen = set()
    taxid_list: List[str] = []
    for t in taxids:
        s = str(t).strip()
        if not s:
            continue
        if s not in seen:
            seen.add(s)
            taxid_list.append(s)

    results: Dict[int, str] = {}

    for batch in _chunked(taxid_list, batch_size):
        handle = Entrez.efetch(db="taxonomy", id=",".join(batch), retmode="xml")
        records = Entrez.read(handle)
        handle.close()

        # records is a list of taxon dicts
        for rec in records:
            try:
                tid = int(rec["TaxId"])
            except Exception:
                continue

            lineage_nodes = []
            for node in rec.get("LineageEx", []):
                rank = (node.get("Rank") or "").lower()
                if rank in RANK_PREFIX:
                    name = node.get("ScientificName", "")
                    if name:
                        lineage_nodes.append(f"{RANK_PREFIX[rank]}{name}")

            if include_self:
                self_rank = (rec.get("Rank") or "").lower()
                if self_rank in RANK_PREFIX:
                    self_name = rec.get("ScientificName", "")
                    if self_name:
                        lineage_nodes.append(f"{RANK_PREFIX[self_rank]}{self_name}")

            # then format like:
            results[tid] = ";".join(lineage_nodes)

        time.sleep(sleep_s)

    # Build series in the original order (including missing)
    idx = [int(x) for x in taxid_list]
    values = [results.get(i, pd.NA) for i in idx]
    return pd.Series(values, index=idx, name="taxonomy")

### Metadata
We need to gather the metadata files for each body site and use the NCBI taxon ID for each OTU to fetch the corresponding taxonomy. Then, we merge the taxonomies with the original metadata. 

In [ ]:
metadata_all = {}

for d, val in directories.items():
    site_dp = os.path.join(gsa_data_dir, d)
    metadata_fp = glob.glob(os.path.join(site_dp, "metadata.tsv"))[0]
    metadata_df = read_metadata(path=metadata_fp)
    
    taxids = set(metadata_df["NCBI_ID"])
    taxa = fetch_taxonomy_series(
        taxids,
        email=email,
        api_key=None,
        include_rank=True,
    )
    metadata_all[val["short"]] = metadata_df.merge(
        right=taxa, left_on="NCBI_ID", right_index=True, how="left"
    )

In [ ]:
metadata_all["air"].head()

In [ ]:
# get the mapping between OTU ID and NCBI ID; remove duplicates

all_meta = pd.concat(metadata_all.values())
print(all_meta.shape)

all_meta = all_meta[~all_meta.index.duplicated()]
print(all_meta.shape)

all_meta["NCBI_ID"] = all_meta["NCBI_ID"].astype(str)
all_meta_dict = all_meta.to_dict()["NCBI_ID"]

### Taxonomy
We process the taxonomies fetched above to get a unique set that we can then import into an artifact.

In [ ]:
taxonomies = []
for k, meta in metadata_all.items():
    taxonomy = meta["taxonomy"].copy()
    taxonomy.index.name = "Feature ID"
    taxonomy.name = "Taxon"
    taxonomies.append(taxonomy)
    
    print(taxonomy.shape)

taxonomy = pd.concat(taxonomies)
taxonomy = taxonomy.fillna("d__Unclassified")
taxonomy.index = taxonomy.index.map(lambda x: all_meta_dict[x])
taxonomy = taxonomy.loc[~taxonomy.index.duplicated("first")]

### Abundance
We process all the provided per-site abundance tables and replace the OTU IDs with NCBI IDs that we got above.

In [ ]:
abundance_dfs = {}

for d, val in directories.items():
    site_dp = os.path.join(gsa_data_dir, d)
    abundances = glob.glob(os.path.join(site_dp, "abundance*"))
    dfs = []
    for ap in abundances:
        sample_no = os.path.splitext(os.path.basename(ap))[0].replace("abundance", "")
        sample_id = val["group"] + "_sample" + sample_no
        df = read_abundance(path=ap, sample=sample_id)
        dfs.append(df)

    bodysite_abundances = pd.concat(dfs, axis=0)
    bodysite_abundances.fillna(0, inplace=True)
    
    # rename OTUs to NCBI IDs
    bodysite_abundances.rename(columns=all_meta_dict, inplace=True)
    bodysite_abundances = bodysite_abundances.T.groupby(level=0).sum().T
    
    abundance_dfs[d] = bodysite_abundances

In [ ]:
abundance_dfs["airways"].head()

### Artifact generation
We will save all the data into artifacts to be used later.

In [ ]:
# taxonomy
#taxonomy.save(os.path.join(gsa_data_dir, "pooled_taxonomy.qza"))

# abundances
#for d, df in abundance_dfs.items():
#    df.save(os.path.join(gsa_data_dir, d, "pooled_abundance.qza"))

## Assembly evaluation
Assemblied obtained for all the 49 samples were analyzed using metaQUAST against the CAMI II pooled reference genomes. Below are parsing the results table returned by multiQC (ran on all of the metaQUAST outputs).

In [ ]:
data_dir = "./data"
body_sites = (
    "airways", "gastrointestinal", "oral","skin", "urogenital"
)

In [ ]:
RANKS = ["d__", "k__", "p__", "c__", "o__", "f__", "g__", "s__"]

def normalize_taxonomy(val):
    # Map the current string into a dictionary: {'o__': 'o__Abc', 'g__': 'g__Ghi'}
    # Note: item[:3] captures the prefix (e.g., 'f__')
    parts_dict = {item[:3]: item for item in val.split(';') if item}
    
    # Reconstruct the string using the dict. If a rank is missing, use the prefix only.
    return ";".join([parts_dict.get(r, r) for r in RANKS])

In [ ]:
quast_results = os.path.join(data_dir, "quast_table.txt")
metadata = os.path.join(data_dir, "metadata.csv")

quast_df = pd.read_csv(quast_results, sep="\t", index_col=0)
meta_df = pd.read_csv(metadata, sep=",", index_col=0)

In [ ]:
quast_df = quast_df.join(meta_df)

In [ ]:
order = ["airways", "gastro", "oral", "skin", "uro"]
present_order = [b for b in order if b in quast_df["bodysite"].unique()]

quast_df_plot = quast_df.copy()
quast_df_plot["Largest contig (Mbp)"] = quast_df_plot["Largest contig (Kbp)"] / 1000

metrics = ["N50 (Kbp)", "L50 (K)", "Largest contig (Mbp)", "ANI"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    sns.boxplot(
        data=quast_df_plot,
        x="bodysite",
        y=metric,
        order=present_order,
        showfliers=False,
        ax=ax
    )
    ax.set_title(metric)
    ax.set_xlabel("bodysite")
    ax.tick_params(axis="x", rotation=45)

    if metric == "N50 (Kbp)":
        ax.set_yscale("log")
        ax.set_ylabel("N50 (Kbp) [log scale]")
    elif metric == "Largest contig (Mbp)":
        ax.set_ylabel("Largest contig (Mbp)")
    else:
        ax.set_ylabel(metric)

fig.tight_layout()
fig.show()

## MAG-based composition evaluation
Feature abundances obtained for all the per-body-site dereplicated MAGs will be evaluated against the expected Golden Standard Assembly abundances from the original THMP dataset.

### Recovered MAGs

In [ ]:
taxonomy_fps, mag_ft_fps = {}, {}
for site in body_sites:
    taxonomy_fps[site] = os.path.join(data_dir, f"{site}/kraken-features-mags-derep-bacteria.qza")
    mag_ft_fps[site] = os.path.join(data_dir, f"{site}/mags-derep-ft-bacteria.qza")

In [ ]:
taxonomy, mag_ft = {}, {}
for site in body_sites:
    taxonomy[site] = q2.Artifact.load(taxonomy_fps[site])
    mag_ft[site] = q2.Artifact.load(mag_ft_fps[site])

In [ ]:
# patch up the taxonomy to replace some of the wrongly labeled levels
taxonomy_edited = {}
for site in body_sites:
    t = taxonomy[site].view(pd.DataFrame)
    t["Taxon"] = t["Taxon"].str.replace(r"d__containing k__.*?;", "d__Bacteria;", regex=True)
    t["Taxon"] = t["Taxon"].str.replace(r"f__containing.*?;", "", regex=True)
    t["Taxon"] = t["Taxon"].str.replace(r"g__containing.*?;", "", regex=True)
    t["Taxon"] = t["Taxon"].apply(normalize_taxonomy)
    taxonomy_edited[site] = q2.Artifact.import_data("FeatureData[Taxonomy]", t)

In [ ]:
# retain only bacteria
for site in body_sites:
    mag_ft[site], = q2_taxa.methods.filter_table(
        table=mag_ft[site],
        taxonomy=taxonomy_edited[site],
        include="d__Bacteria",
    )   

In [ ]:
# collapse all the taxa to level 8
mag_ft_lvl8 = {}
for site in body_sites:
    mag_ft_lvl8[site], = q2_taxa.methods.collapse(
        table=mag_ft[site],
        taxonomy=taxonomy_edited[site],
        level=8
    )

In [ ]:
# convert feature tables to relative abundances
mag_ft_lvl8_rel = {}
for site in body_sites:
    mag_ft_lvl8_rel[site], = feature_table.methods.relative_frequency(
        table=mag_ft_lvl8[site]
    )

In [ ]:
# merge all
mag_ft_lvl8_rel_all, = feature_table.methods.merge(
    tables=list(mag_ft_lvl8_rel.values()),
)
taxonomy_edited_all, = feature_table.methods.merge_taxa(
    data=list(taxonomy_edited.values()),
)

### References

In [ ]:
# copy the GSA data

# taxonomy
# shutil.copy(
#     os.path.join(gsa_data_dir, "pooled_taxonomy.qza"),
#     os.path.join(data_dir, "pooled-taxonomy.qza")
# )

# abundances
# for site in body_sites:
#     shutil.copy(
#         os.path.join(gsa_data_dir, site, "pooled_abundance.qza"),
#         os.path.join(data_dir, site, "pooled-abundance.qza")
#     )

In [ ]:
taxonomy_ref = q2.Artifact.load(os.path.join(data_dir, "pooled-taxonomy.qza"))
abundance_ref = {}
for site in body_sites:
    abundance_ref[site] = q2.Artifact.load(os.path.join(data_dir, f"{site}/pooled-abundance.qza"))

In [ ]:
# normalize taxonomy
taxonomy_df = taxonomy_ref.view(pd.DataFrame)
taxonomy_df["Taxon"] = taxonomy_df["Taxon"].apply(normalize_taxonomy)
taxonomy_ref = q2.Artifact.import_data("FeatureData[Taxonomy]", taxonomy_df)

In [ ]:
# retain only bacteria
for site in body_sites:
    abundance_ref[site], = q2_taxa.methods.filter_table(
        table=abundance_ref[site],
        taxonomy=taxonomy_ref,
        include="d__Bacteria",
    )   

In [ ]:
# collapse all the taxa to level 8
ref_ft_lvl8 = {}
for site in body_sites:
    ref_ft_lvl8[site], = q2_taxa.methods.collapse(
        table=abundance_ref[site],
        taxonomy=taxonomy_ref,
        level=8
    )

In [ ]:
# convert feature tables to relative abundances
ref_ft_lvl8_rel = {}
for site in body_sites:
    ref_ft_lvl8_rel[site], = feature_table.methods.relative_frequency(
        table=ref_ft_lvl8[site]
    )

In [ ]:
# merge all
ref_ft_lvl8_rel_all, = feature_table.methods.merge(
    tables=list(ref_ft_lvl8_rel.values()),
)

### Evaluate composition

In [ ]:
comparison, = quality_control.visualizers.evaluate_composition(
    expected_features=ref_ft_lvl8_rel_all,
    observed_features=mag_ft_lvl8_rel_all,
    depth=7,
    plot_observed_features_ratio=False,
)

In [ ]:
comparison.save("composition-comparison.qzv")